# ML Pipeline 4: Donor Amount Uplift (180 days)
Predictive + explanatory models for identifying donors likely to increase giving.

In [ ]:
import os, sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.db import load_env, build_engine
from src.features import build_frame, split_xy
from src.modeling import train_predictive_and_explanatory


## Business KPI Mapping

- Stakeholder owner: Fundraising Director
- Decision enabled: prioritize donors for higher-touch ask strategy
- Primary KPI: uplift in donated amount among targeted donors
- Guardrail KPIs: donor fatigue/opt-out rate, outreach cost per additional peso raised
- Minimum success criteria: +10% incremental donation amount at <=12% increase in outreach cost

## Problem Framing
- Predictive: classify donors likely to increase donation amount within 180 days.
- Explanatory: identify donor characteristics associated with amount uplift.
- Caveat: this is a proxy for "would give more if asked"; no direct intervention flag exists.

In [ ]:
load_env('.env')
engine = build_engine(os.getenv('DB_PROFILE', 'local'))
df = build_frame(engine)
X, y = split_xy(df)
print(df.shape, 'uplift rate=', round(y.mean(),4))
df.head()

In [ ]:
# Data Understanding Audit: missingness + anomaly checks
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(10).to_frame('missing_pct'))

audit = {
    'rows': len(df),
    'uplift_positive_rate': float(y.mean()),
    'negative_donation_rows': int((df['donation_value'] < 0).sum()),
    'amount_to_date_lt_snapshot_rows': int((df['amount_to_date'] < df['donation_value']).sum()),
}
print('Audit summary:', audit)

print('Feature rationale: donor history + profile features tie directly to expected capacity/propensity to increase giving.')

In [ ]:
import numpy as np
import subprocess
import sys

try:
    import seaborn as sns
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'seaborn', 'matplotlib'])
    import seaborn as sns
    import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['donation_value'], bins=15, kde=True, ax=axes[0])
axes[0].set_title('Snapshot Donation Value Distribution')

sns.boxplot(data=df, x='supporter_type', y='amount_to_date', ax=axes[1])
axes[1].set_title('Cumulative Amount by Supporter Type')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
rf, rf_metrics, logit, logit_metrics, coef_df = train_predictive_and_explanatory(X, y)
print('Predictive RF metrics:', {k: round(v,4) for k,v in rf_metrics.items()})
print('Explanatory logistic metrics:', {k: round(v,4) for k,v in logit_metrics.items()})
display(coef_df.head(12))
display(coef_df.tail(12).sort_values('coefficient'))

In [ ]:
# Threshold tuning + FP/FN cost table for donor uplift targeting
uplift_proba_full = logit.predict_proba(X)[:, 1]
thresholds = np.arange(0.1, 0.95, 0.05)
fp_cost = 1.0
fn_cost = 2.5
rows = []
for t in thresholds:
    pred = (uplift_proba_full >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    total_cost = fp_cost * fp + fn_cost * fn
    rows.append({'threshold': round(float(t), 2), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn), 'total_cost': float(total_cost)})

cost_df = pd.DataFrame(rows).sort_values('total_cost').reset_index(drop=True)
display(cost_df.head(10))
best_t = float(cost_df.loc[0, 'threshold'])
print(f'Selected threshold by cost minimization: {best_t:.2f}')

## Evaluation In Business Terms
- False positives: outreach to donors unlikely to increase giving.
- False negatives: missed opportunities for tailored asks.
Use uplift scores to prioritize who receives premium ask strategies.

## Operationalization Policy + Monitoring

- Threshold policy: choose minimum-cost threshold from FP/FN table; default fallback 0.50.
- Action bands: high uplift probability -> premium ask, medium -> standard ask, low -> nurture-only touchpoints.
- Retraining cadence: monthly retrain, with early retrain when PR-AUC degrades >15% or feature drift alerts fire.
- Monitoring references:
  - `ml-pipelines/integration/pipeline_registry.yaml`
  - `ml-pipelines/integration/monitoring_spec.md`
  - `ml-pipelines/integration/README.md`

## Causal Caveat
Associations do not prove asking caused uplift; validate with controlled campaigns.

## Deployment Notes
Serve donor uplift probability in a fundraiser dashboard and rank donor segments for outreach planning.